# Проект №3. Тема: Исследование г. Пскова. Итория границ. Объекты культурного наследия. Исторические постройки. 

## Используемые данные
- Перечень объектов культурного наследия, федерального и регионального значения. Источник: Портал открытые данные министерства культуры РФ https://opendata.mkrf.ru/ 
- Печень жилых домов с годом постройки. Источник: Фонд развития территорий https://фрт.рф/  
- Исторические границы города в разный период истории. Источник: http://www.etomesto.ru/ 

# Этапы работы 

### Выгружаем библиотеки

In [212]:
import osmnx as ox
import geopandas as gpd
import pandas as pd
import folium

### Отображаем границы города. Существующие и исторические

In [243]:
# Современная граница города Пскова
pskov = ox.geocode_to_gdf("городской округ Псков, Псковская область, Россия")
pskov = pskov.to_crs(epsg=4326)

In [244]:
# Исторические границы
kreml = gpd.read_file("granitsi/Терртория кремля_G1.gpkg")
wall_1309 = gpd.read_file("granitsi/Стена города_1309_G2.gpkg")
wall_1375 = gpd.read_file("granitsi/Стена города_1375_G3.gpkg")
wall_1399_1466 = gpd.read_file("granitsi/Стена города_1399_1466_G4.gpkg")

# Переводим в EPSG:4326
kreml = kreml.to_crs(epsg=4326)
wall_1309 = wall_1309.to_crs(epsg=4326)
wall_1375 = wall_1375.to_crs(epsg=4326)
wall_1399_1466 = wall_1399_1466.to_crs(epsg=4326)

In [245]:
# Центр карты
center = pskov.geometry.centroid.iloc[0]

# Создаем карту
m = folium.Map(
    location=[center.y, center.x],
    zoom_start=13,
    tiles="CartoDB Positron",
    control_scale=True
)

# Современная граница города
folium.GeoJson(
    pskov,
    name="Современная граница города Пскова",
    style_function=lambda feature: {
        "color": "red",
        "weight": 3,
        "fillColor": "none",
        "fillOpacity": 0
    }
).add_to(m)

# Территория кремля
folium.GeoJson(
    kreml,
    name="Территория кремля",
    style_function=lambda feature: {
        "color": "purple",
        "weight": 5
    }
).add_to(m)

# Стена 1309
folium.GeoJson(
    wall_1309,
    name="Стена города 1309",
    style_function=lambda feature: {
        "color": "orange",
        "weight": 5
    }
).add_to(m)

# Стена 1375
folium.GeoJson(
    wall_1375,
    name="Стена города 1375",
    style_function=lambda feature: {
        "color": "green",
        "weight": 5
    }
).add_to(m)

# Стена 1399–1466
folium.GeoJson(
    wall_1399_1466,
    name="Стена города 1399–1466",
    style_function=lambda feature: {
        "color": "blue",
        "weight": 5
    }
).add_to(m)



C:\Users\Pilyukova.ss\AppData\Local\Temp\ipykernel_5364\1062544401.py:2: UserWarning: Geometry is in a geographic CRS. Results from 'centroid' are likely incorrect. Use 'GeoSeries.to_crs()' to re-project geometries to a projected CRS before this operation.

  center = pskov.geometry.centroid.iloc[0]


### Отображаем объекты культурного наследия


In [246]:
%pip install openpyxl

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 24.0 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [247]:
# Загружаем объекты культурного наследия
okn = pd.read_excel("ОКН_Псков_c коорд.xlsx")

# Посмотрим первые строки
okn.head()

,Id,Объект,Номер в реестре,Регион,address,Latitude,Longitude,Категория историко-культурного значения,Вид объекта,Принадлежность к Юнеско,Особо ценный объект,Принадлежность к ВОВ,coords
0,1,ШТАЛАГ 372» - территория концентрационного лаг...,602131000000000,Псковская область,"Псковская область, г. Псков, территория между ...",57.808792,28.293342,Регионального значения,Достопримечательное место,нет,нет,Нет,NaN
1,2,Ансамбль Вознесенского монастыря,601520000000000,Псковская область,"Псковская область, г. Псков, Советская ул., 64-а",57.809629,28.338503,Федерального значения,Ансамбль,нет,нет,Нет,NaN
2,3,Ансамбль Кремля,601520000000000,Псковская область,"Псковская область, г. Псков, Кремль",57.819971,28.330017,Федерального значения,Ансамбль,нет,нет,Нет,NaN
3,4,Ансамбль Снетогорского монастыря,601520000000000,Псковская область,"Псковская область, г. Псков, ул. Снятная гора,...",57.834872,28.263413,Федерального значения,Ансамбль,нет,нет,Нет,NaN
4,5,Ансамбль Спасо-Мирожского монастыря,611320000000000,Псковская область,"Псковская область, г. Псков, Мирожский монасты...",57.806885,28.328563,Федерального значения,Ансамбль,нет,нет,Нет,NaN


In [248]:
# Посмотрим, какие категории есть в таблице
okn["Категория историко-культурного значения"].value_counts()

Категория историко-культурного значения
Федерального значения     95
Регионального значения    58
Name: count, dtype: int64

In [249]:
from folium.plugins import MarkerCluster

In [250]:
okn_federal = okn_city[
    okn_city["Категория историко-культурного значения"] == "Федерального значения"
]

okn_regional = okn_city[
    okn_city["Категория историко-культурного значения"] == "Регионального значения"
]

In [251]:
# Создаем кластеры
federal_cluster = MarkerCluster(name="ОКН федерального значения")
regional_cluster = MarkerCluster(name="ОКН регионального значения")

# Федеральные ОКН
for _, row in okn_federal.iterrows():
    folium.CircleMarker(
        location=[row["Latitude"], row["Longitude"]],
        radius=6,
        color="gold",
        fill=True,
        fill_color="gold",
        fill_opacity=0.9,
        popup=f"<b>{row['Объект']}</b><br>{row['address']}",
        tooltip=row["Объект"]
    ).add_to(federal_cluster)

# Региональные ОКН
for _, row in okn_regional.iterrows():
    folium.CircleMarker(
        location=[row["Latitude"], row["Longitude"]],
        radius=6,
        color="green",
        fill=True,
        fill_color="green",
        fill_opacity=0.9,
        popup=f"<b>{row['Объект']}</b><br>{row['address']}",
        tooltip=row["Объект"]
    ).add_to(regional_cluster)

# Добавляем кластеры на карту
federal_cluster.add_to(m)
regional_cluster.add_to(m)

In [252]:
okn_gdf = gpd.GeoDataFrame(
    okn,
    geometry=gpd.points_from_xy(okn["Longitude"], okn["Latitude"]),
    crs="EPSG:4326"
)

pskov = pskov.to_crs(epsg=4326)

okn_city = gpd.sjoin(
    okn_gdf,
    pskov[["geometry"]],
    how="inner",
    predicate="within"
)

okn_federal = okn_city[
    okn_city["Категория историко-культурного значения"] == "Федерального значения"
]

okn_regional = okn_city[
    okn_city["Категория историко-культурного значения"] == "Регионального значения"
]

print(len(okn_city))
print(len(okn_federal))
print(len(okn_regional))

152
94
58


In [254]:
federal_cluster = MarkerCluster(name="ОКН федерального значения")
regional_cluster = MarkerCluster(name="ОКН регионального значения")

for _, row in okn_federal.iterrows():
    folium.CircleMarker(
        location=[row["Latitude"], row["Longitude"]],
        radius=6,
        color="purple",
        weight=2,
        fill=True,
        fill_color="purple",
        fill_opacity=0.9,
        popup=f"<b>{row['Объект']}</b><br>{row['address']}",
        tooltip=row["Объект"]
    ).add_to(federal_cluster)

for _, row in okn_regional.iterrows():
    folium.CircleMarker(
        location=[row["Latitude"], row["Longitude"]],
        radius=6,
        color="teal",
        weight=2,
        fill=True,
        fill_color="teal",
        fill_opacity=0.9,
        popup=f"<b>{row['Объект']}</b><br>{row['address']}",
        tooltip=row["Объект"]
    ).add_to(regional_cluster)

federal_cluster.add_to(m)
regional_cluster.add_to(m)

In [255]:
okn_city[okn_city["Longitude"] > 28.8][
    ["Объект", "address", "Latitude", "Longitude"]
]

,Объект,address,Latitude,Longitude


In [256]:
print(m._children.keys())

odict_keys(['cartodbpositron', 'geo_json_07ec429a443fec82af9a80b51baf9eb5', 'geo_json_8d24548e78b4e56d6f142a27fd5dc3fa', 'geo_json_c0ebcd1df820028ac3c8b284c707fd3e', 'geo_json_92e7418b15bad2258b1f56328e6daca8', 'geo_json_efdc61d32d4886a2950cec7787797fea', 'marker_cluster_5510e2a153b01c9514125eb81aedce9d', 'marker_cluster_544709cfabc3f784f0221ad18e876752', 'marker_cluster_3573bfaec9da44afe6a633a137ab450e', 'marker_cluster_b1e2abb065dd0d8e9ef00f3193821aa3', 'marker_cluster_4316751f086f9e5bce0f2101deb94064', 'marker_cluster_1850139164b929463b390930bee3da1e'])


### Жилые дома с годом постройки

In [257]:
# Загружаем жилые дома
houses = pd.read_csv("vse_doma_pskov.csv")

# Переименовываем столбец с годом постройки
houses = houses.rename(columns={"Unnamed: 4": "Год постройки"})

# Оставляем только нужные столбцы
houses = houses[["adress", "Latitude", "Longitude", "Год постройки"]]

houses.head()

,adress,Latitude,Longitude,Год постройки
0,"обл. Псковская, г. Псков, пер. 1-й Хлебной Гор...",57.823798,28.352246,1958
1,"обл. Псковская, г. Псков, пер. 1-й Хлебной Гор...",57.824145,28.351141,1958
2,"обл. Псковская, г. Псков, ул. 1-я Поселочная, ...",57.796607,28.344980,1947
3,"обл. Псковская, г. Псков, ул. 1-я Поселочная, ...",57.796821,28.345991,1940
4,"обл. Псковская, г. Псков, ул. 1-я Поселочная, ...",57.797128,28.347200,1959


In [258]:
houses["Год постройки"].describe()

count    1399.000000
mean     1976.944246
std        24.753111
min      1694.000000
25%      1963.000000
50%      1975.000000
75%      1993.000000
max      2021.000000
Name: Год постройки, dtype: float64

In [259]:
# Дома по историческим периодам
houses_old = houses[houses["Год постройки"] <= 1917]

houses_middle = houses[
    (houses["Год постройки"] > 1917) &
    (houses["Год постройки"] <= 1945)
]

houses_new = houses[houses["Год постройки"] > 1945]

print("До 1917:", len(houses_old))
print("1918–1945:", len(houses_middle))
print("После 1945:", len(houses_new))

До 1917: 46
1918–1945: 16
После 1945: 1337


In [260]:
# Слои для жилых домов
houses_old_layer = folium.FeatureGroup(name="Дома до 1917 года")
houses_middle_layer = folium.FeatureGroup(name="Дома 1918–1945 гг.")

# Дома до 1917 года
for _, row in houses_old.iterrows():
    folium.CircleMarker(
        location=[row["Latitude"], row["Longitude"]],
        radius=5,
        color="brown",
        fill=True,
        fill_color="brown",
        fill_opacity=0.8,
        popup=f"<b>{row['adress']}</b><br>Год постройки: {int(row['Год постройки'])}"
    ).add_to(houses_old_layer)

# Дома 1918–1945 гг.
for _, row in houses_middle.iterrows():
    folium.CircleMarker(
        location=[row["Latitude"], row["Longitude"]],
        radius=5,
        color="orange",
        fill=True,
        fill_color="orange",
        fill_opacity=0.8,
        popup=f"<b>{row['adress']}</b><br>Год постройки: {int(row['Год постройки'])}"
    ).add_to(houses_middle_layer)

# Добавляем слои домов на карту
houses_old_layer.add_to(m)
houses_middle_layer.add_to(m)

In [261]:
from folium.plugins import HeatMap, Fullscreen, MiniMap, MousePosition

In [262]:
# Дома после 1945 года для тепловой карты
heat_data = houses_new[["Latitude", "Longitude"]].dropna().values.tolist()

# Добавляем тепловую карту
HeatMap(
    heat_data,
    name="Плотность домов после 1945 года",
    radius=15,
    blur=20,
    min_opacity=0.3
).add_to(m)

In [263]:
# Интерактивные инструменты
Fullscreen().add_to(m)

MiniMap(
    toggle_display=True
).add_to(m)

MousePosition().add_to(m)

In [264]:
folium.LayerControl(collapsed=False).add_to(m)

m

# Проект №3. Тема: Исследование г. Пскова. Итория границ. Объекты культурного наследия. Исторические постройки. 

## Используемые данные
- Перечень объектов культурного наследия, федерального и регионального значения. Источник: Портал открытые данные министерства культуры РФ https://opendata.mkrf.ru/ 
- Печень жилых домов с годом постройки. Источник: Фонд развития территорий https://фрт.рф/  
- Исторические границы города в разный период истории. Источник: http://www.etomesto.ru/ 

# Этапы работы 

### Выгружаем библиотеки

In [ ]:
import osmnx as ox
import geopandas as gpd
import pandas as pd
import folium

### Отображаем границы города. Существующие и исторические

In [ ]:
# Современная граница города Пскова
pskov = ox.geocode_to_gdf("городской округ Псков, Псковская область, Россия")
pskov = pskov.to_crs(epsg=4326)

In [ ]:
# Исторические границы
kreml = gpd.read_file("granitsi/Терртория кремля_G1.gpkg")
wall_1309 = gpd.read_file("granitsi/Стена города_1309_G2.gpkg")
wall_1375 = gpd.read_file("granitsi/Стена города_1375_G3.gpkg")
wall_1399_1466 = gpd.read_file("granitsi/Стена города_1399_1466_G4.gpkg")

# Переводим в EPSG:4326
kreml = kreml.to_crs(epsg=4326)
wall_1309 = wall_1309.to_crs(epsg=4326)
wall_1375 = wall_1375.to_crs(epsg=4326)
wall_1399_1466 = wall_1399_1466.to_crs(epsg=4326)

In [ ]:
# Центр карты
center = pskov.geometry.centroid.iloc[0]

# Создаем карту
m = folium.Map(
    location=[center.y, center.x],
    zoom_start=13,
    tiles="CartoDB Positron",
    control_scale=True
)

# Современная граница города
folium.GeoJson(
    pskov,
    name="Современная граница города Пскова",
    style_function=lambda feature: {
        "color": "red",
        "weight": 3,
        "fillColor": "none",
        "fillOpacity": 0
    }
).add_to(m)

# Территория кремля
folium.GeoJson(
    kreml,
    name="Территория кремля",
    style_function=lambda feature: {
        "color": "purple",
        "weight": 5
    }
).add_to(m)

# Стена 1309
folium.GeoJson(
    wall_1309,
    name="Стена города 1309",
    style_function=lambda feature: {
        "color": "orange",
        "weight": 5
    }
).add_to(m)

# Стена 1375
folium.GeoJson(
    wall_1375,
    name="Стена города 1375",
    style_function=lambda feature: {
        "color": "green",
        "weight": 5
    }
).add_to(m)

# Стена 1399–1466
folium.GeoJson(
    wall_1399_1466,
    name="Стена города 1399–1466",
    style_function=lambda feature: {
        "color": "blue",
        "weight": 5
    }
).add_to(m)



C:\Users\Pilyukova.ss\AppData\Local\Temp\ipykernel_5364\1062544401.py:2: UserWarning: Geometry is in a geographic CRS. Results from 'centroid' are likely incorrect. Use 'GeoSeries.to_crs()' to re-project geometries to a projected CRS before this operation.

  center = pskov.geometry.centroid.iloc[0]


### Отображаем объекты культурного наследия


In [ ]:
%pip install openpyxl

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 24.0 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [ ]:
# Загружаем объекты культурного наследия
okn = pd.read_excel("ОКН_Псков_c коорд.xlsx")

# Посмотрим первые строки
okn.head()

,Id,Объект,Номер в реестре,Регион,address,Latitude,Longitude,Категория историко-культурного значения,Вид объекта,Принадлежность к Юнеско,Особо ценный объект,Принадлежность к ВОВ,coords
0,1,ШТАЛАГ 372» - территория концентрационного лаг...,602131000000000,Псковская область,"Псковская область, г. Псков, территория между ...",57.808792,28.293342,Регионального значения,Достопримечательное место,нет,нет,Нет,NaN
1,2,Ансамбль Вознесенского монастыря,601520000000000,Псковская область,"Псковская область, г. Псков, Советская ул., 64-а",57.809629,28.338503,Федерального значения,Ансамбль,нет,нет,Нет,NaN
2,3,Ансамбль Кремля,601520000000000,Псковская область,"Псковская область, г. Псков, Кремль",57.819971,28.330017,Федерального значения,Ансамбль,нет,нет,Нет,NaN
3,4,Ансамбль Снетогорского монастыря,601520000000000,Псковская область,"Псковская область, г. Псков, ул. Снятная гора,...",57.834872,28.263413,Федерального значения,Ансамбль,нет,нет,Нет,NaN
4,5,Ансамбль Спасо-Мирожского монастыря,611320000000000,Псковская область,"Псковская область, г. Псков, Мирожский монасты...",57.806885,28.328563,Федерального значения,Ансамбль,нет,нет,Нет,NaN


In [ ]:
# Посмотрим, какие категории есть в таблице
okn["Категория историко-культурного значения"].value_counts()

Категория историко-культурного значения
Федерального значения     95
Регионального значения    58
Name: count, dtype: int64

In [ ]:
from folium.plugins import MarkerCluster

In [ ]:
okn_federal = okn_city[
    okn_city["Категория историко-культурного значения"] == "Федерального значения"
]

okn_regional = okn_city[
    okn_city["Категория историко-культурного значения"] == "Регионального значения"
]

In [ ]:
# Создаем кластеры
federal_cluster = MarkerCluster(name="ОКН федерального значения")
regional_cluster = MarkerCluster(name="ОКН регионального значения")

# Федеральные ОКН
for _, row in okn_federal.iterrows():
    folium.CircleMarker(
        location=[row["Latitude"], row["Longitude"]],
        radius=6,
        color="gold",
        fill=True,
        fill_color="gold",
        fill_opacity=0.9,
        popup=f"<b>{row['Объект']}</b><br>{row['address']}",
        tooltip=row["Объект"]
    ).add_to(federal_cluster)

# Региональные ОКН
for _, row in okn_regional.iterrows():
    folium.CircleMarker(
        location=[row["Latitude"], row["Longitude"]],
        radius=6,
        color="green",
        fill=True,
        fill_color="green",
        fill_opacity=0.9,
        popup=f"<b>{row['Объект']}</b><br>{row['address']}",
        tooltip=row["Объект"]
    ).add_to(regional_cluster)

# Добавляем кластеры на карту
federal_cluster.add_to(m)
regional_cluster.add_to(m)

### Жилые дома с годом постройки

In [ ]:
# Загружаем жилые дома
houses = pd.read_csv("vse_doma_pskov.csv")

# Переименовываем столбец с годом постройки
houses = houses.rename(columns={"Unnamed: 4": "Год постройки"})

# Оставляем только нужные столбцы
houses = houses[["adress", "Latitude", "Longitude", "Год постройки"]]

houses.head()

,adress,Latitude,Longitude,Год постройки
0,"обл. Псковская, г. Псков, пер. 1-й Хлебной Гор...",57.823798,28.352246,1958
1,"обл. Псковская, г. Псков, пер. 1-й Хлебной Гор...",57.824145,28.351141,1958
2,"обл. Псковская, г. Псков, ул. 1-я Поселочная, ...",57.796607,28.344980,1947
3,"обл. Псковская, г. Псков, ул. 1-я Поселочная, ...",57.796821,28.345991,1940
4,"обл. Псковская, г. Псков, ул. 1-я Поселочная, ...",57.797128,28.347200,1959


In [ ]:
houses["Год постройки"].describe()

count    1399.000000
mean     1976.944246
std        24.753111
min      1694.000000
25%      1963.000000
50%      1975.000000
75%      1993.000000
max      2021.000000
Name: Год постройки, dtype: float64

In [ ]:
# Дома по историческим периодам
houses_old = houses[houses["Год постройки"] <= 1917]

houses_middle = houses[
    (houses["Год постройки"] > 1917) &
    (houses["Год постройки"] <= 1945)
]

houses_new = houses[houses["Год постройки"] > 1945]

print("До 1917:", len(houses_old))
print("1918–1945:", len(houses_middle))
print("После 1945:", len(houses_new))

До 1917: 46
1918–1945: 16
После 1945: 1337


In [ ]:
# Слои для жилых домов
houses_old_layer = folium.FeatureGroup(name="Дома до 1917 года")
houses_middle_layer = folium.FeatureGroup(name="Дома 1918–1945 гг.")

# Дома до 1917 года
for _, row in houses_old.iterrows():
    folium.CircleMarker(
        location=[row["Latitude"], row["Longitude"]],
        radius=5,
        color="brown",
        fill=True,
        fill_color="brown",
        fill_opacity=0.8,
        popup=f"<b>{row['adress']}</b><br>Год постройки: {int(row['Год постройки'])}"
    ).add_to(houses_old_layer)

# Дома 1918–1945 гг.
for _, row in houses_middle.iterrows():
    folium.CircleMarker(
        location=[row["Latitude"], row["Longitude"]],
        radius=5,
        color="orange",
        fill=True,
        fill_color="orange",
        fill_opacity=0.8,
        popup=f"<b>{row['adress']}</b><br>Год постройки: {int(row['Год постройки'])}"
    ).add_to(houses_middle_layer)

# Добавляем слои домов на карту
houses_old_layer.add_to(m)
houses_middle_layer.add_to(m)

In [ ]:
from folium.plugins import HeatMap, Fullscreen, MiniMap, MousePosition

In [ ]:
# Дома после 1945 года для тепловой карты
heat_data = houses_new[["Latitude", "Longitude"]].dropna().values.tolist()

# Добавляем тепловую карту
HeatMap(
    heat_data,
    name="Плотность домов после 1945 года",
    radius=15,
    blur=20,
    min_opacity=0.3
).add_to(m)

In [ ]:
# Интерактивные инструменты
Fullscreen().add_to(m)

MiniMap(
    toggle_display=True
).add_to(m)

MousePosition().add_to(m)

In [ ]:
folium.LayerControl(collapsed=False).add_to(m)

m

In [242]:
center = pskov.geometry.centroid.iloc[0]

m = folium.Map(
    location=[center.y, center.x],
    zoom_start=11,
    tiles="CartoDB Positron"
)

C:\Users\Pilyukova.ss\AppData\Local\Temp\ipykernel_5364\2220710428.py:1: UserWarning: Geometry is in a geographic CRS. Results from 'centroid' are likely incorrect. Use 'GeoSeries.to_crs()' to re-project geometries to a projected CRS before this operation.

  center = pskov.geometry.centroid.iloc[0]
